# Exploring the tracking data

Analysis only: no labels, no model, no submission. Everything here is computed from the automatic
tracking and the Smart Tagging; the ground truth is not used.

1. **Slots and teams** — which slot is which team, how slots are used, how many dots per frame
2. **Speed fields** — what speed information the file contains and whether it can be trusted
3. **Ball visibility** — visible stretches, gaps, ball speed, where the ball gets lost
4. **Dead-ball candidates from the tracking** — ball out, stationary ball, collective slowdown
5. **Smart Tagging set pieces** — what each set-piece type looks like in the tracking
6. **Cross-check** — how tracking candidates and tags agree, and what each one misses
7. **Conclusions**

All statistics are over **graded frames** (the two halves) unless stated otherwise.

## 0. Setup

In [ ]:
import io
import os
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = os.path.abspath('.')
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, 'starter'))

import games
import load_smart_tagging
from baseline import team_names, team_of_each_track
from utils import draw_pitch

warnings.filterwarnings('ignore', category=RuntimeWarning)  # nan-means over empty slices
pd.set_option('display.max_rows', 60)

GAME = games.GAME
FPS = games.GAMES[GAME]['fps']
N = games.GAMES[GAME]['n_video_frames']
L, W = games.GAMES[GAME]['pitch']
HALVES = games.half_video_blocks(GAME)
KICKOFF_2ND = 113742  # FAQ.md: the 2nd half is graded from 110,808 but kicks off here

graded = np.zeros(N, bool)
for _, first, last in HALVES:
    graded[first:last + 1] = True
half_of = np.zeros(N, int)
for half, first, last in HALVES:
    half_of[first:last + 1] = half

H, A = 0, 1
TEAM_NAME = {H: 'Barça', A: 'Athletic'}
TEAM_COLOUR = {H: 'tab:blue', A: 'tab:red'}


def runs_of(mask):
    # [(start, end_exclusive)] of the True runs of a boolean array.
    edges = np.diff(np.r_[0, mask.astype(np.int8), 0])
    return list(zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1)))


def pct(x, qs=(10, 50, 90)):
    return ', '.join(f'p{q} {np.nanpercentile(x, q):.1f}' for q in qs)


print('halves:', HALVES)

## 1. Load the tracking

Raw line: `frame_id:slot;slot;…;slot:ball`, every object `x,y,s` normalised 0–1 (speed `s` in
m/s). Replacing `:` and `;` by `,` turns each line into one flat CSV row. Cached in
`data/cache/` (git-ignored).

In [ ]:
CACHE = os.path.join(games.DATA_DIR, 'cache', 'atd_explore.npz')


def load_tracking(force=False):
    if os.path.isfile(CACHE) and not force:
        with np.load(CACHE) as z:
            return {k: z[k] for k in z.files}

    meta_path, raw_path = games.require(['atd_meta', 'atd_raw'])
    with open(raw_path) as handle:
        text = handle.read().replace(':', ',').replace(';', ',')
    table = pd.read_csv(io.StringIO(text), header=None, dtype=np.float64).to_numpy()
    del text

    n_slots = (table.shape[1] - 4) // 3
    assert len(table) == N and np.array_equal(table[:, 0], np.arange(1, N + 1))
    scale = np.array([L, W])
    players = table[:, 1:1 + 3 * n_slots].reshape(N, n_slots, 3)
    data = {
        'ball_xy': table[:, -3:-1] * scale,
        'ball_s_file': table[:, -1],
        'player_xy': (players[:, :, :2] * scale).astype(np.float32),
        'player_s': players[:, :, 2].astype(np.float32),
        'team': team_of_each_track(meta_path),
    }
    os.makedirs(os.path.dirname(CACHE), exist_ok=True)
    np.savez(CACHE, **data)
    return data


trk = load_tracking()
ball_xy, ball_s_file = trk['ball_xy'], trk['ball_s_file']
player_xy, player_s, team = trk['player_xy'], trk['player_s'], trk['team']
active = ~np.isnan(player_xy[:, :, 0])
ball_vis = ~np.isnan(ball_xy[:, 0])
print(f'{N} frames, {player_xy.shape[1]} slots; teams:', team_names(games.asset_paths()['atd_meta']))

## 2. Slots and teams

Each slot belongs to one team for the whole match (fixed in the XML). A slot holds one
continuous **track piece** at a time; when the tracker loses someone, the piece ends, and the
person reappears later in some free slot of the same team.

In [ ]:
rows = []
for s in range(player_xy.shape[1]):
    pieces = runs_of(active[:, s] & graded)
    lens = np.array([b - a for a, b in pieces]) / FPS
    rows.append({
        'slot': s + 1,
        'team': TEAM_NAME[team[s]],
        'active_share': active[graded, s].mean(),
        'pieces': len(pieces),
        'median_piece_s': np.median(lens) if len(lens) else np.nan,
        'longest_piece_s': lens.max() if len(lens) else np.nan,
    })
slots = pd.DataFrame(rows).set_index('slot')

print('slots per team:', slots['team'].value_counts().to_dict())
print('slot numbers per team:')
for t in (H, A):
    print(f'  {TEAM_NAME[t]:<8}', list(slots.index[slots['team'] == TEAM_NAME[t]]))
slots.round(2)

In [ ]:
all_lens = []
for s in range(player_xy.shape[1]):
    all_lens += [(b - a) / FPS for a, b in runs_of(active[:, s] & graded)]
all_lens = np.array(all_lens)
print(f'{len(all_lens)} track pieces on graded frames; length (s): {pct(all_lens)}, max {all_lens.max():.0f}')
for limit in [1, 5, 30, 300]:
    print(f'  pieces shorter than {limit:>3} s: {np.mean(all_lens < limit):.0%}')

Two slots are active almost all match (24 for Barça, 46 for Athletic). Where they stand in each
half shows what they are:

In [ ]:
for s in (24, 46):
    for half, first, last in HALVES:
        x = player_xy[first:last + 1, s - 1, 0]
        print(f'slot {s} ({TEAM_NAME[team[s - 1]]}), half {half}: active {np.mean(~np.isnan(x)):.0%}, '
              f'x p5-p95 {np.nanpercentile(x, 5):.0f}-{np.nanpercentile(x, 95):.0f} m (pitch is 0-{L:.0f})')

### Dots per frame, per team

In [ ]:
inside = active & (player_xy[:, :, 0] >= -1) & (player_xy[:, :, 0] <= L + 1) \
                & (player_xy[:, :, 1] >= -1) & (player_xy[:, :, 1] <= W + 1)

table = {}
for t in (H, A):
    on = team == t
    n_all = active[graded][:, on].sum(axis=1)
    n_in = inside[graded][:, on].sum(axis=1)
    table[TEAM_NAME[t]] = {
        'mean dots': n_all.mean(),
        '% frames >11 dots': 100 * np.mean(n_all > 11),
        '% frames >11 inside pitch (±1 m)': 100 * np.mean(n_in > 11),
        '% frames <10 dots': 100 * np.mean(n_all < 10),
        '% frames <8 dots': 100 * np.mean(n_all < 8),
    }
pd.DataFrame(table).round(1)

In [ ]:
# Where are the dots that are outside the pitch? (sample every 5th graded frame)
sample = np.flatnonzero(graded)[::5]
fig, ax = plt.subplots(figsize=(9, 6))
draw_pitch(ax, L, W)
for t in (H, A):
    xy = player_xy[sample][:, team == t].reshape(-1, 2)
    xy = xy[~np.isnan(xy[:, 0])]
    out = (xy[:, 0] < -1) | (xy[:, 0] > L + 1) | (xy[:, 1] < -1) | (xy[:, 1] > W + 1)
    ax.scatter(xy[out, 0], xy[out, 1], s=1, alpha=0.2, color=TEAM_COLOUR[t], label=TEAM_NAME[t])
    print(f'{TEAM_NAME[t]}: {out.mean():.1%} of dots are outside the pitch')
ax.set_xlim(-10, L + 10)
ax.set_ylim(-12, W + 12)
ax.legend(loc='upper right', markerscale=10)
ax.set_title('Dots outside the pitch (sampled graded frames)')
plt.show()

In [ ]:
# Duplicates: two dots of the same team closer than 1 m (sample every 25th graded frame)
dup = {H: 0, A: 0}
frames = np.flatnonzero(graded)[::25]
for f in frames:
    for t in (H, A):
        xy = player_xy[f][(team == t) & active[f]]
        if len(xy) < 2:
            continue
        d = np.linalg.norm(xy[:, None] - xy[None], axis=2)
        np.fill_diagonal(d, np.inf)
        dup[t] += d.min() < 1.0
for t in (H, A):
    print(f'{TEAM_NAME[t]}: two dots < 1 m apart on {dup[t] / len(frames):.1%} of sampled frames')

## 3. Speed fields

The file has two speed fields: one per player slot, one for the ball. Both are compared with the
speed computed from positions (centred difference over ±2 frames).

In [ ]:
def speed_from_positions(xy, k=2):
    # Centred difference over +/- k frames, m/s. Works for [N, 2] and [N, S, 2].
    v = np.full(xy.shape[:-1], np.nan, dtype=np.float32)
    v[k:-k] = np.linalg.norm(xy[2 * k:] - xy[:-2 * k], axis=-1) * FPS / (2 * k)
    return v


player_v = speed_from_positions(player_xy)
both = graded[:, None] & ~np.isnan(player_s) & ~np.isnan(player_v)
a, b = player_s[both], player_v[both]
print(f'player speed field present on {np.mean(~np.isnan(player_s[graded][active[graded]])):.1%} of active dots')
print(f'player speed field (m/s): {pct(a, (50, 90, 99))}, max {a.max():.1f}')
print(f'computed from positions : {pct(b, (50, 90, 99))}')
print(f'correlation {np.corrcoef(a, b)[0, 1]:.3f}, median |difference| {np.median(np.abs(a - b)):.2f} m/s')

In [ ]:
ball_v = speed_from_positions(ball_xy)
has_file = ~np.isnan(ball_s_file)
print('ball speed field present on graded frames:')
print(f'  with a ball position    : {np.sum(has_file & ball_vis & graded)}')
print(f'  without a ball position : {np.sum(has_file & ~ball_vis & graded)}')
print(f'  ball position present   : {np.sum(ball_vis & graded)}')
ok = has_file & ~np.isnan(ball_v) & graded
if ok.sum():
    print(f'where both exist ({ok.sum()} frames): correlation {np.corrcoef(ball_s_file[ok], ball_v[ok])[0, 1]:.3f}, '
          f'median |difference| {np.median(np.abs(ball_s_file[ok] - ball_v[ok])):.2f} m/s')

## 4. Ball visibility

Visible stretches and gaps, ball speed while visible, and where the ball gets lost.

In [ ]:
vis_runs = runs_of(ball_vis & graded)
gap_runs = runs_of(~ball_vis & graded)
vis_len = np.array([b - a for a, b in vis_runs]) / FPS
gap_len = np.array([b - a for a, b in gap_runs]) / FPS

print(f'ball visible on {ball_vis[graded].mean():.1%} of graded frames')
print(f'visible stretches: {len(vis_runs)}, length (s): {pct(vis_len)}, max {vis_len.max():.0f}')
print(f'gaps             : {len(gap_runs)}, length (s): {pct(gap_len)}, max {gap_len.max():.0f}')
for limit in [1, 5, 10, 30]:
    print(f'  missing time in gaps longer than {limit:>2} s: {gap_len[gap_len > limit].sum() / gap_len.sum():.0%}')

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
bins = np.logspace(-2, 3, 40)
axes[0].hist(vis_len, bins=bins, color='tab:green')
axes[0].set_title('visible stretch length')
axes[1].hist(gap_len, bins=bins, color='tab:grey')
axes[1].set_title('gap length')
for ax in axes:
    ax.set_xscale('log')
    ax.set_xlabel('seconds')
plt.tight_layout()
plt.show()

In [ ]:
# Visibility over the match, per minute
fig, ax = plt.subplots(figsize=(13, 3))
for half, first, last in HALVES:
    idx = np.arange(first, last + 1)
    minute = (idx - first) // (60 * FPS)
    share = pd.Series(ball_vis[idx]).groupby(minute).mean()
    ax.plot(share.index + (0 if half == 1 else 50), share.values, marker='.', label=f'half {half}')
ax.set_ylabel('share of frames with ball')
ax.set_xlabel('minute from start of half (2nd half shifted by 50)')
ax.set_ylim(0, 1)
ax.legend()
plt.show()

In [ ]:
v = ball_v[ball_vis & graded]
v = v[~np.isnan(v)]
print(f'ball speed while visible (m/s): {pct(v, (10, 50, 90, 99))}')
for lo, hi in [(0, 0.5), (0.5, 3), (3, 8), (8, 15), (15, 25), (25, 60)]:
    print(f'  {lo:>4}-{hi:<3} m/s: {np.mean((v >= lo) & (v < hi)):.0%}')

fig, ax = plt.subplots(figsize=(8, 3))
ax.hist(np.clip(v, 0, 40), bins=80, color='tab:green')
ax.set_xlabel('ball speed from positions (m/s), clipped at 40')
plt.show()

### Where and when does the ball get lost?

Compare the last frame before each gap with ordinary visible frames: position on the pitch,
how crowded it is around the ball, and how fast the ball was moving.

In [ ]:
def players_near(f_idx, xy, radius):
    d = np.linalg.norm(player_xy[f_idx] - xy[:, None, :], axis=2)
    return np.sum(d < radius, axis=1)


def backward_speed(idx, k=4):
    # Speed over the k frames *before* idx, so a frame right before a gap still has one.
    return np.linalg.norm(ball_xy[idx] - ball_xy[idx - k], axis=1) * FPS / k


last_seen = np.array([a - 1 for a, b in gap_runs if a - 1 >= 4 and ball_vis[a - 1]])
reference = np.flatnonzero(ball_vis & graded)[::50]

compare = {}
for name, idx in [('last frame before a gap', last_seen), ('ordinary visible frame', reference)]:
    xy = ball_xy[idx]
    d_line = np.minimum.reduce([xy[:, 0], L - xy[:, 0], xy[:, 1], W - xy[:, 1]])
    compare[name] = {
        'frames': len(idx),
        'players within 3 m (median)': np.median(players_near(idx, xy, 3)),
        'players within 5 m (mean)': players_near(idx, xy, 5).mean(),
        'ball speed over previous 4 frames, m/s (median)': np.nanmedian(backward_speed(idx)),
        '% ball faster than 15 m/s': 100 * np.nanmean(backward_speed(idx) > 15),
        '% within 2 m of a line or outside': 100 * np.mean(d_line < 2),
        '% outside the pitch': 100 * np.mean(d_line < 0),
    }
pd.DataFrame(compare).round(1)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for ax, (name, idx) in zip(axes, [('ordinary visible frames', reference), ('last seen before a gap', last_seen)]):
    draw_pitch(ax, L, W)
    ax.scatter(ball_xy[idx, 0], ball_xy[idx, 1], s=4, alpha=0.4, color='black')
    ax.set_xlim(-8, L + 8)
    ax.set_ylim(-8, W + 8)
    ax.set_title(name)
plt.show()

## 4b. Dead-ball candidates from the tracking

Signals that can be computed without any label:
- **ball out**: ball visibly more than 0.5 m past a line;
- **stationary ball**: visible and slower than 0.5 m/s for at least 1 s;
- **collective slowdown**: the mean speed of all players (file speed field), averaged over 1 s.

In [ ]:
d_line = np.minimum.reduce([ball_xy[:, 0], L - ball_xy[:, 0], ball_xy[:, 1], W - ball_xy[:, 1]])
out = ball_vis & (d_line < -0.5) & graded

# Episodes: out frames, merged when separated by less than 2 s.
out_eps = []
for a, b in runs_of(out):
    if out_eps and a - out_eps[-1][1] < 2 * FPS:
        out_eps[-1] = (out_eps[-1][0], b)
    else:
        out_eps.append((a, b))

rows = []
for a, b in out_eps:
    x, y = ball_xy[a]
    side = 'goal line' if (x < 0 or x > L) else 'touchline'
    nxt_in = np.flatnonzero(ball_vis[b:b + 120 * FPS] & (d_line[b:b + 120 * FPS] > 0.5))
    rows.append({'start': a, 'side': side, 'visible_out_s': (b - a) / FPS,
                 'seen_inside_again_after_s': nxt_in[0] / FPS if len(nxt_in) else np.nan})
out_df = pd.DataFrame(rows)
print(f'{len(out_eps)} ball-out episodes on graded frames:', out_df['side'].value_counts().to_dict())
print(f"visible while out (s): {pct(out_df['visible_out_s'])}")
print(f"until seen inside the pitch again (s): {pct(out_df['seen_inside_again_after_s'])}")

In [ ]:
stationary = ball_vis & graded & (ball_v < 0.5)
stat_eps = [(a, b) for a, b in runs_of(stationary) if b - a >= FPS]
stat_len = np.array([b - a for a, b in stat_eps]) / FPS
stat_xy = np.array([ball_xy[(a + b) // 2] for a, b in stat_eps])
print(f'{len(stat_eps)} stationary-ball episodes (>= 1 s), length (s): {pct(stat_len)}')

fig, ax = plt.subplots(figsize=(9, 6))
draw_pitch(ax, L, W)
sc = ax.scatter(stat_xy[:, 0], stat_xy[:, 1], c=np.clip(stat_len, 1, 15), s=15, cmap='viridis')
plt.colorbar(sc, ax=ax, label='duration (s, clipped at 15)')
ax.set_xlim(-8, L + 8)
ax.set_ylim(-8, W + 8)
ax.set_title('Where the ball sits still for at least 1 s')
plt.show()

In [ ]:
mean_speed = np.nanmean(np.where(active, player_s, np.nan), axis=1)
mean_speed_1s = pd.Series(mean_speed).rolling(FPS, center=True, min_periods=1).mean().to_numpy()
print(f'mean player speed, 1 s average (m/s): {pct(mean_speed_1s[graded], (5, 25, 50, 75, 95))}')

slow = graded & (mean_speed_1s < 1.5)
slow_eps = [(a, b) for a, b in runs_of(slow) if b - a >= 3 * FPS]
slow_len = np.array([b - a for a, b in slow_eps]) / FPS
print(f'{len(slow_eps)} slow episodes (mean player speed < 1.5 m/s for >= 3 s), '
      f'length (s): {pct(slow_len)}, total {slow_len.sum() / 60:.1f} min')

pre_ko = np.arange(HALVES[1][1], KICKOFF_2ND)
print(f'for reference, before the 2nd-half kickoff (known dead): mean player speed {np.nanmean(mean_speed[pre_ko]):.2f} m/s, '
      f'ball visible {ball_vis[pre_ko].mean():.0%}')

### What happens around a ball-out?

Every ball-out episode lined up at the frame the ball is first seen out (time 0): ball speed,
mean player speed, and the share of episodes where the ball is visible. Lines are medians
(share for visibility), bands the middle 50%.

In [ ]:
def peri(signal, anchors, before_s, after_s):
    offs = np.arange(-int(before_s * FPS), int(after_s * FPS) + 1)
    M = np.full((len(anchors), len(offs)), np.nan)
    for i, f in enumerate(anchors):
        idx = f + offs
        ok = (idx >= 0) & (idx < N)
        M[i, ok] = signal[idx[ok]]
    return offs / FPS, M


def plot_peri(ax, t, M, label, colour, share=False):
    if share:
        ax.plot(t, np.nanmean(M, axis=0), color=colour, label=label)
        return
    med = np.nanmedian(M, axis=0)
    q1, q3 = np.nanpercentile(M, 25, axis=0), np.nanpercentile(M, 75, axis=0)
    ax.plot(t, med, color=colour, label=label)
    ax.fill_between(t, q1, q3, color=colour, alpha=0.15)


def peri_panel(groups, before_s, after_s, title):
    fig, axes = plt.subplots(1, 3, figsize=(16, 3.8), sharex=True)
    for (label, anchors), colour in zip(groups.items(), plt.cm.tab10.colors):
        t, M = peri(ball_v, anchors, before_s, after_s)
        plot_peri(axes[0], t, M, f'{label} ({len(anchors)})', colour)
        t, M = peri(mean_speed_1s, anchors, before_s, after_s)
        plot_peri(axes[1], t, M, label, colour)
        t, M = peri(ball_vis.astype(float), anchors, before_s, after_s)
        plot_peri(axes[2], t, M, label, colour, share=True)
    axes[0].set_ylabel('ball speed (m/s)')
    axes[0].set_ylim(0, 30)
    axes[1].set_ylabel('mean player speed (m/s)')
    axes[2].set_ylabel('share with ball visible')
    axes[2].set_ylim(0, 1)
    for ax in axes:
        ax.axvline(0, color='black', lw=0.6)
        ax.set_xlabel('seconds from anchor')
    axes[0].legend(fontsize=8)
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


peri_panel({side: out_df.loc[out_df['side'] == side, 'start'].to_numpy() for side in ['touchline', 'goal line']},
           10, 40, 'Around the first frame the ball is seen out of the pitch')

## 5. Smart Tagging set pieces

Each set-piece tag is a ~5 s window around a restart, with a `Type`. Below: what each type looks
like in the tracking (anchored at the tag's start frame), where the ball sits, and when the
restart actually happens relative to the tag.

In [ ]:
events = load_smart_tagging.load()
sp = pd.DataFrame([{
    'type': e['tags'].get('Type'),
    'team': e['tags'].get('Team'),
    'start': e['start_frame'],
    'end': e['end_frame'],
} for e in events if (e['code'] or '').strip().upper() == 'SET PIECES'])
sp['graded'] = graded[np.clip(sp['start'], 0, N - 1)]
print(sp.groupby('type').agg(count=('start', 'size'), graded=('graded', 'sum'),
                             length_s=('start', lambda s: ((sp.loc[s.index, 'end'] - s) / FPS).median())))

In [ ]:
types = ['Throw-in', 'Corner Kick', 'Goal Kick', 'Free Kick', 'Kick Off (After Goal)']
peri_panel({t: sp.loc[(sp['type'] == t) & sp['graded'], 'start'].to_numpy() for t in types},
           20, 25, 'Around the start of each set-piece tag')

In [ ]:
# Inside each tag window (+5 s after it): where is the ball when it is slowest, and when does it
# get struck (first frame > 5 m/s after a frame < 1 m/s)?
rows = []
for _, r in sp[sp['graded']].iterrows():
    win = np.arange(r['start'], min(r['end'] + 5 * FPS, N))
    v = ball_v[win]
    if np.all(np.isnan(v)):
        rows.append({'type': r['type'], 'ball_seen': False})
        continue
    slowest = win[np.nanargmin(v)]
    still = np.flatnonzero(v < 1)
    strike = np.nan
    if len(still):
        after = np.flatnonzero(v[still[0]:] > 5)
        if len(after):
            strike = (still[0] + after[0]) / FPS
    rows.append({'type': r['type'], 'ball_seen': True, 'x': ball_xy[slowest, 0], 'y': ball_xy[slowest, 1],
                 'restart_after_tag_start_s': strike})
inwin = pd.DataFrame(rows)
print(inwin.groupby('type').agg(tags=('ball_seen', 'size'), ball_seen=('ball_seen', 'sum'),
                                restart_s_median=('restart_after_tag_start_s', 'median'),
                                restart_found=('restart_after_tag_start_s', 'count')))

fig, ax = plt.subplots(figsize=(9, 6))
draw_pitch(ax, L, W)
for (t, g), colour in zip(inwin[inwin['ball_seen']].groupby('type'), plt.cm.tab10.colors):
    ax.scatter(g['x'], g['y'], s=25, color=colour, label=t)
ax.set_xlim(-8, L + 8)
ax.set_ylim(-8, W + 8)
ax.legend(fontsize=8, loc='upper right')
ax.set_title('Ball position at its slowest inside each set-piece tag')
plt.show()

## 6. Cross-check: tracking candidates vs tags

In [ ]:
def near(anchors, targets, lo_s, hi_s):
    # For each anchor: is there a target starting within [anchor + lo, anchor + hi]?
    targets = np.sort(np.asarray(targets))
    out = []
    for f in anchors:
        i = np.searchsorted(targets, f + lo_s * FPS)
        out.append(i < len(targets) and targets[i] <= f + hi_s * FPS)
    return np.array(out, dtype=bool)


sp_g = sp[sp['graded']]
out_starts = out_df['start'].to_numpy()
out_df['tag_within_40s'] = near(out_starts, sp_g['start'], -10, 40)
print('ball-out episodes followed by any set-piece tag (−10 s .. +40 s):')
print(out_df.groupby('side')['tag_within_40s'].agg(['size', 'sum', 'mean']).round(2))

sp_g = sp_g.assign(out_before=[np.any((out_starts >= s - 60 * FPS) & (out_starts <= s + 10 * FPS))
                               for s in sp_g['start']])
print('\nset-piece tags with a ball-out episode in the 60 s before (or 10 s after) the tag:')
print(sp_g.groupby('type')['out_before'].agg(['size', 'sum', 'mean']).round(2))

In [ ]:
slow_starts = np.array([a for a, b in slow_eps])
explained = []
for a, b in slow_eps:
    has_out = np.any((out_starts >= a - 30 * FPS) & (out_starts <= b))
    has_tag = np.any((sp_g['start'].to_numpy() >= a - 30 * FPS) & (sp_g['start'].to_numpy() <= b + 10 * FPS))
    explained.append((has_out, has_tag))
explained = np.array(explained)
print(f'{len(slow_eps)} slow episodes (mean player speed < 1.5 m/s for >= 3 s):')
print(f'  with a ball-out shortly before or during : {explained[:, 0].mean():.0%}')
print(f'  with a set-piece tag nearby               : {explained[:, 1].mean():.0%}')
print(f'  with neither (possible whistle stoppages) : {np.mean(~explained[:, 0] & ~explained[:, 1]):.0%} '
      f'({np.sum(~explained[:, 0] & ~explained[:, 1])} episodes)')

### One stretch of play with everything on it

Ten minutes of the second half after kickoff: mean player speed, ball visibility, ball-out
episodes and set-piece tags on one time axis.

In [ ]:
def overview(first, minutes):
    last = first + minutes * 60 * FPS
    t = np.arange(first, last)
    x = (t - first) / FPS / 60
    fig, ax = plt.subplots(figsize=(16, 4))
    ax.plot(x, mean_speed_1s[t], color='black', lw=0.8, label='mean player speed (1 s)')
    ax.fill_between(x, 0, 0.4, where=ball_vis[t], color='tab:green', alpha=0.4, label='ball visible')
    for a, b in out_eps:
        if first <= a < last:
            ax.axvspan((a - first) / FPS / 60, (b - first) / FPS / 60 + 0.02, color='tab:orange', alpha=0.6)
    for _, r in sp.iterrows():
        if first <= r['start'] < last:
            ax.axvspan((r['start'] - first) / FPS / 60, (r['end'] - first) / FPS / 60, color='tab:purple', alpha=0.2)
            ax.text((r['start'] - first) / FPS / 60, 6.6, r['type'], fontsize=7, rotation=90, va='top')
    ax.plot([], [], color='tab:orange', lw=6, label='ball out')
    ax.plot([], [], color='tab:purple', lw=6, alpha=0.3, label='set-piece tag')
    ax.set_ylim(0, 7)
    ax.set_xlabel(f'minutes from frame {first}')
    ax.set_ylabel('m/s')
    ax.legend(loc='upper right', fontsize=8, ncol=4)
    plt.tight_layout()
    plt.show()


overview(KICKOFF_2ND, 10)

## 7. Conclusions

### Slots and teams
- **Slots 1–24 are Barça, 25–46 Athletic.** Slots 22, 23 and 44 are never used.
- **Slot 24 is Barça's goalkeeper** for the whole match (one end in each half). **Slot 46 is
  mostly Athletic's goalkeeper**, occasionally reused for someone else.
- Every other slot is reused many times: 1,219 track pieces on graded frames, median 35 s, 18%
  shorter than 5 s. Apart from the goalkeepers, **slot identity carries no information**.
- About 10.2 dots per team per frame. More than 11 on 8% (Barça) / 5% (Athletic) of frames, and
  almost all of those extra dots are **inside** the pitch; two same-team dots < 1 m apart on ~7%
  of frames (the same player tracked twice). Fewer than 10 dots on 13–15% of frames.
  Dots outside the pitch are rare (0.3%) and sit along the lines.

### Speeds
- **The player speed field is reliable**: present on 92% of active dots, correlation 0.93 with the
  speed computed from positions. It can be used directly.
- **The ball speed field is useless**: correlation 0.06 with the real ball movement, and half of
  its values sit on frames with no ball position. Ball speed has to be computed from positions.
- Visible ball speed: median 5 m/s, 90% below 14 m/s, 99% below 24 m/s.

### Ball visibility
- Visible on 59% of graded frames, in 149 stretches (median 14 s) separated by 150 gaps
  (median 11 s). **85% of the missing time is in gaps longer than 10 s, 44% in gaps longer than
  30 s**, and there are multi-minute blackouts (visibility per minute drops to ~0 several times).
  Interpolating across gaps cannot work; the missing ball is a long-gap problem.
- **Where the ball gets lost:** not in crowds (players within 3 m: same as usual) and not in fast
  flight (only 2% faster than 15 m/s, vs 8% normally). The ball is **slower than usual** right
  before a gap (median 2.0 vs 5.2 m/s) and **more often at or beyond a line** (29% vs 9%).
  The tracker tends to lose a slow ball near the lines — at a player's feet, in hands for a
  throw-in, or at a stoppage. **The start of a gap may itself be a dead-ball hint** (to test).

### Dead-ball signals from the tracking
- **Ball out:** 67 episodes (51 touchline, 16 goal line). The ball is seen out only briefly
  (median 0.9 s). After a goal-line exit the ball mostly disappears for 10–20 s and the restart
  comes ~20 s later; after a touchline exit it mostly stays visible and play resumes sooner.
- **Stationary ball (≥ 1 s):** only 42 episodes, but they sit exactly on set-piece spots —
  corners, the 6-yard box, the touchlines, the centre spot. Precise when present, rare because the
  ball is usually invisible during stoppages.
- **Collective slowdown is weak on its own.** Median mean-player speed is 1.5 m/s; before the
  2nd-half kickoff (known dead) it is 1.27 m/s; around a ball-out it dips by only ~0.1–0.3 m/s.
  A 1.5 m/s threshold flags 48 minutes, mostly ordinary play. The clearer signal is the
  **acceleration after a restart**: players go from ~1 to ~2.5 m/s within ~10 s.

### Smart Tagging set pieces
- Set-piece windows are **35 s long** (kickoffs 5 s). Their **start sits close to the restart**:
  during the ~10 s before the start, players are slow (~1 m/s) and the ball is still or missing;
  within a few seconds after it, the ball is struck and players accelerate. So the **dead ball
  lies before the tag start**, and the 35 s window is the play after the restart.
- The automatic restart finder in section 5 is noisy (medians 5–20 s after the tag start); the
  averaged plots are the more trustworthy reading.
- **Coverage, both ways:** 88% of throw-in tags follow a visible ball-out, but there are 51
  touchline exits for 26 throw-in tags, and 16 goal-line exits for 22 corner + goal-kick tags.
  Each source misses stoppages the other sees; they need to be combined.
- The 4 kick-offs after a goal have no visible ball-out before them (the ball is lost before it
  crosses the line).
- Of 157 slow-play episodes, 39% have neither a ball-out nor a tag nearby: candidate whistle
  stoppages, but the threshold is loose, so many will be slow live play.

### What this means for the approach
1. Use team membership and positions, never slot identity (except the two goalkeeper slots).
2. Use the player speed field; compute ball speed from positions.
3. Over 40% of the match the ball is missing in long gaps: possession there has to come from
   what the players do, not from the ball.
4. Dead ball needs several signals together — ball out, stationary ball on a set-piece spot,
   gap starting near a line, set-piece tag start (dead before it), acceleration after a restart.
   Collective slowdown alone is not enough.
5. Set-piece tags mark the **end** of a stoppage, which helps time the dead → live change.

### Open questions for the next analysis
- What happens during the multi-minute blackouts (camera, zoom, a tracker failure)?
- How often does a gap that starts near a line or with a slow ball turn out to be a stoppage?
- A better restart detector (stationary ball → struck) to time the end of stoppages precisely.
- Are the extra dots inside the pitch the referee being assigned to a team?